In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType
)
import time
import requests
from datetime import datetime
from pyspark.sql.functions import col, countDistinct

spark = SparkSession.builder \
    .master("spark://group13-master:7077") \
    .appName("Hive_test") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.shuffle.service.enabled", "false") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

parquet_path = "hdfs:///project/reddit/parquet/part1"

26/03/19 13:11:05 WARN StandaloneSchedulerBackend: Dynamic allocation enabled without spark.executor.cores explicitly set, you may get more executors allocated than expected. It's recommended to set spark.executor.cores explicitly. Please check SPARK-30299 for more details.


In [18]:
def run_analysis(data, target):

    start = time.time()

    # spark.sparkContext.setJobDescription("1. Filter target subreddit users")
    users = (
        data.filter(col("subreddit") == target)
        .select("author")
        .distinct()
    )

    # spark.sparkContext.setJobDescription("2. Joining")
    user_posts = data.join(users, "author")

    # spark.sparkContext.setJobDescription("3. Find related subreddits")
    other_posts = user_posts.filter(col("subreddit") != target)

    result = (
        other_posts.groupBy("subreddit")
        .agg(countDistinct("author").alias("unique_users"))
    )

    # spark.sparkContext.setJobDescription("4. Collect")
    result.collect()

    runtime = time.time() - start

    return runtime

In [20]:
# EXECUTE ONLY ONCE
# Number of buckets?
# df = spark.read.parquet(parquet_path)
# df.write.format("parquet").mode("overwrite").bucketBy(25, "author").sortBy("subreddit").option("compression", "snappy").saveAsTable("parquet_data")

26/03/19 13:18:28 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/03/19 13:18:29 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/03/19 13:18:29 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/19 13:18:29 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/03/19 13:18:33 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


In [21]:
st = time.time()
df_parquet = spark.table("parquet_data")
# df_parquet = spark.read.parquet(parquet_path)
print(time.time() - st)

0.9711272716522217


In [22]:
# Keeep neccesary columns
clean_df = df_parquet.select("author", "subreddit") \
    .filter(col("author").isNotNull()) \
    .filter(col("subreddit").isNotNull()) \
    .filter(col("author") != "[deleted]")

In [23]:
spark.catalog.clearCache()
spark.sql("CLEAR CACHE")
run_analysis(clean_df, "AskReddit")

92.33056282997131

In [ ]:
# Or use the REST API after each job
apps = requests.get("http://localhost:4040/api/v1/applications").json()
app_id = apps[0]["id"]
jobs = requests.get(f"http://localhost:4040/api/v1/applications/{app_id}/jobs").json()

for job in jobs:
    fmt = "%Y-%m-%dT%H:%M:%S.%f%Z"
    submission = datetime.strptime(job["submissionTime"], fmt)
    completion = datetime.strptime(job["completionTime"], fmt)
    duration_ms = (completion - submission).total_seconds() * 1000
    print(f"Job {job['jobId']}: name={job.get('description', 'no name')}, duration={duration_ms}ms, stages={len(job['stageIds'])}")
    

In [24]:
spark.stop()